# Task 6: ETL Pipeline in Google Colab to Oracle ADW

This notebook implements a complete ETL workflow using Google Colab.

- **Extract**: Download cleaned CSV datasets from public cloud URLs.
- **Transform**: Clean, normalize, and enrich datasets with ETL metadata.
- **Load**: Insert data into Oracle Autonomous Data Warehouse (ADW).
- **Validate**: Run SQL checks to verify row counts and loaded data.

## Step 1: Install Dependencies
Install required Python libraries in Colab.

In [ ]:
%pip install -q pandas sqlalchemy oracledb requests

## Step 2: Extract Data from Public Cloud Source
Set public raw CSV URLs (for example, GitHub Raw links) and download to Colab local storage.

This notebook is preconfigured to use your GitHub repository raw URLs.

In [ ]:
from pathlib import Path
import requests

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_CSV_URLS = {
    'cleaned_aisles.csv': 'https://raw.githubusercontent.com/basdilhan/DW_MarketBasket/main/cleaned_aisles.csv',
    'cleaned_departments.csv': 'https://raw.githubusercontent.com/basdilhan/DW_MarketBasket/main/cleaned_departments.csv',
    'cleaned_products.csv': 'https://raw.githubusercontent.com/basdilhan/DW_MarketBasket/main/cleaned_products.csv',
    'cleaned_orders.csv': 'https://raw.githubusercontent.com/basdilhan/DW_MarketBasket/main/cleaned_orders.csv'
}

def download_csvs(url_map):
    local_paths = {}
    for file_name, url in url_map.items():
        print(f'[Extract] Downloading {file_name} from: {url}')
        response = requests.get(url, timeout=120)
        response.raise_for_status()

        target_path = DATA_DIR / file_name
        target_path.write_bytes(response.content)
        print(f'[Extract] Saved to: {target_path} ({target_path.stat().st_size:,} bytes)')
        local_paths[file_name] = target_path

    return local_paths

csv_paths = download_csvs(PUBLIC_CSV_URLS)
print('[Extract] All files downloaded successfully.')

## Step 3: Upload and Configure Oracle Wallet in Colab
Upload your wallet zip file (for example, `Wallet_DWBI2026.zip`) when prompted.

In [ ]:
import os
import re
import zipfile
from google.colab import files

print('[Wallet] Upload wallet zip file now...')
uploaded = files.upload()

wallet_zip_name = next((name for name in uploaded.keys() if name.lower().endswith('.zip')), None)
if wallet_zip_name is None:
    raise FileNotFoundError('No .zip wallet file uploaded. Please upload your wallet zip file.')

wallet_dir = Path('/content/wallet')
wallet_dir.mkdir(parents=True, exist_ok=True)

print(f'[Wallet] Extracting {wallet_zip_name} to {wallet_dir} ...')
with zipfile.ZipFile(wallet_zip_name, 'r') as zf:
    zf.extractall(wallet_dir)

sqlnet_path = wallet_dir / 'sqlnet.ora'
if sqlnet_path.exists():
    txt = sqlnet_path.read_text(encoding='utf-8')
    txt = re.sub(r'DIRECTORY="[^"]*"', f'DIRECTORY="{wallet_dir}"', txt)
    sqlnet_path.write_text(txt, encoding='utf-8')
    print(f'[Wallet] Updated WALLET_LOCATION in: {sqlnet_path}')

os.environ['TNS_ADMIN'] = str(wallet_dir)
print(f'[Wallet] TNS_ADMIN set to: {os.environ["TNS_ADMIN"]}')

print('[Wallet] Extracted files:')
for item in sorted(wallet_dir.glob('*')):
    print(f' - {item.name}')

## Step 4: Diagnose Network and Connect to Oracle ADW
Use wallet-based authentication and validate network reachability from Colab before connecting.

If you see `DPY-6000` / `ORA-12506`, Oracle listener is rejecting the client by service ACL policy. In OCI, add the current Colab public IP to the database access control list (or temporarily allow `0.0.0.0/0` for testing).

## Step 4a: Wallet and Network Diagnostic
Check wallet aliases, detect current Colab public IP, and test TCP reachability to the ADW endpoint.

In [ ]:
import os
import re
import socket
import requests

print('[Diagnose] Checking wallet configuration...')
print(f'[Diagnose] TNS_ADMIN={os.environ.get("TNS_ADMIN")}')
print(f'[Diagnose] Wallet directory exists: {wallet_dir.exists()}')

tnsnames_path = wallet_dir / 'tnsnames.ora'
aliases = []
selected_alias = 'dwbi2026_high'
host = None
port = None

if tnsnames_path.exists():
    print('[Diagnose] Found tnsnames.ora')
    tnsnames_content = tnsnames_path.read_text(encoding='utf-8')

    # Collect top-level alias names
    for line in tnsnames_content.split('\n'):
        line = line.strip()
        if '=' in line and not line.startswith('#'):
            alias_name = line.split('=')[0].strip()
            if alias_name and not any(c in alias_name for c in ['(', ')']):
                aliases.append(alias_name)

    print('[Diagnose] TNS aliases defined:')
    for a in aliases:
        print(f'  - {a}')

    # Parse host/port for the selected alias by slicing from alias start to next alias
    alias_start = re.search(rf'(?im)^\s*{re.escape(selected_alias)}\s*=', tnsnames_content)
    if alias_start:
        tail = tnsnames_content[alias_start.end():]
        next_alias = re.search(r'(?im)^\s*[A-Za-z0-9_]+\s*=', tail)
        block = tail[:next_alias.start()] if next_alias else tail

        host_match = re.search(r'HOST\s*=\s*([^)\s]+)', block, re.IGNORECASE)
        port_match = re.search(r'PORT\s*=\s*([^)\s]+)', block, re.IGNORECASE)

        if host_match:
            host = host_match.group(1)
        if port_match:
            try:
                port = int(port_match.group(1))
            except ValueError:
                port = None

        if host and port:
            print(f'[Diagnose] Parsed endpoint for {selected_alias}: host={host}, port={port}')
        else:
            print(f'[Diagnose] Alias found but host/port parse incomplete for {selected_alias}')
    else:
        print(f'[Diagnose] Could not find descriptor block for alias: {selected_alias}')
else:
    print(f'[Diagnose] tnsnames.ora NOT found at {tnsnames_path}')

print('[Diagnose] Wallet files:')
for item in sorted(wallet_dir.glob('*')):
    size_kb = item.stat().st_size / 1024 if item.is_file() else 0
    print(f'  - {item.name} ({size_kb:.1f} KB)')

# Detect public IP of current Colab runtime
colab_public_ip = None
try:
    colab_public_ip = requests.get('https://api.ipify.org', timeout=10).text.strip()
    print(f'[Diagnose] Colab public IP: {colab_public_ip}')
except Exception as ip_error:
    print(f'[Diagnose] Unable to determine public IP: {ip_error}')

# DNS + TCP reachability check to ADW host
if host and port:
    try:
        resolved_ip = socket.gethostbyname(host)
        print(f'[Diagnose] DNS resolved {host} -> {resolved_ip}')
    except Exception as dns_error:
        print(f'[Diagnose] DNS resolution failed for {host}: {dns_error}')

    try:
        with socket.create_connection((host, port), timeout=8):
            print(f'[Diagnose] TCP connectivity OK to {host}:{port}')
    except Exception as tcp_error:
        print(f'[Diagnose] TCP connectivity FAILED to {host}:{port}: {tcp_error}')

print('[Diagnose] --- ACTIONS IF DPY-6000 / ORA-12506 CONTINUES ---')
print('1. In OCI, open your Autonomous Database -> Network Access / Access control rules.')
print('2. Add this Colab IP to allowed list as CIDR /32 (example: 203.0.113.10/32).')
print('3. Or temporarily allow 0.0.0.0/0 for testing, then tighten it again.')
print('4. Stop and Start the database once after ACL changes, then retry Step 4b.')
if colab_public_ip:
    print(f'[Diagnose] Suggested CIDR to add now: {colab_public_ip}/32')

## Step 4b: Connect to ADW
Attempt connection using wallet aliases in thin mode. If all aliases fail with `DPY-6000` / `ORA-12506`, update OCI access control rules for the current Colab IP and retry.

In [ ]:
from getpass import getpass
import oracledb
from sqlalchemy import create_engine, text

db_user = 'ADMIN'
db_password = getpass('Enter ADW password: ')

# Try aliases in preferred order. Keep these aligned with wallet tnsnames.ora.
service_aliases = [
    'dwbi2026_high',
    'dwbi2026_medium',
    'dwbi2026_low',
    'dwbi2026_tp',
    'dwbi2026_tpurgent'
]

print('[Connect] Using thin mode with wallet (no Instant Client).')
print(f'[Connect] Wallet directory: {wallet_dir}')

# Make wallet directory explicit for thin mode alias resolution.
oracledb.defaults.config_dir = str(wallet_dir)

working_alias = None
last_error = None

for alias in service_aliases:
    print(f'[Connect] Testing alias: {alias}')
    try:
        with oracledb.connect(
            user=db_user,
            password=db_password,
            dsn=alias,
            config_dir=str(wallet_dir)
        ) as conn:
            with conn.cursor() as cur:
                cur.execute('SELECT 1 FROM dual')
                cur.fetchone()
        working_alias = alias
        print(f'[Connect] Alias {alias} is reachable.')
        break
    except Exception as conn_error:
        last_error = conn_error
        print(f'[Connect] Alias {alias} failed: {conn_error}')

if working_alias is None:
    error_msg = str(last_error) if last_error else 'Unknown connection failure'
    print('[Connect] No service alias worked.')

    if 'DPY-6000' in error_msg or 'ORA-12506' in error_msg or 'Listener refused' in error_msg:
        raise RuntimeError(
            'Service ACL filtering is blocking this Colab runtime IP. '
            'In OCI, add current Colab public IP as /32 in Autonomous DB access control list '
            '(or temporarily allow 0.0.0.0/0 for testing), then Stop/Start DB and retry.'
        )

    raise RuntimeError(f'Connection failed for all aliases. Last error: {error_msg}')

print(f'[Connect] Creating SQLAlchemy engine with alias: {working_alias}')
engine = create_engine(
    'oracle+oracledb://',
    connect_args={
        'user': db_user,
        'password': db_password,
        'dsn': working_alias,
        'config_dir': str(wallet_dir)
    },
    pool_pre_ping=True
)

print('[Connect] Verifying SQLAlchemy engine...')
with engine.connect() as conn:
    conn.execute(text('SELECT 1 FROM dual'))
print('[Connect] Connection successful.')

## Step 5: Transform Data
Define reusable data-cleaning and Oracle dtype mapping functions.

In [ ]:
import pandas as pd
from sqlalchemy.dialects.oracle import NUMBER, VARCHAR2, TIMESTAMP

def clean_dimension_df(df, table_name):
    before_rows = len(df)
    df = df.drop_duplicates().copy()

    object_cols = df.select_dtypes(include=['object']).columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.strip()

    df['etl_processed_timestamp'] = pd.Timestamp.now()
    print(f'[Transform] {table_name}: rows before={before_rows:,}, after={len(df):,}')
    return df

def normalize_numeric_columns(df_in):
    df_out = df_in.copy()
    float_cols = df_out.select_dtypes(include=['float64', 'float32']).columns

    for col in float_cols:
        non_null = df_out[col].dropna()
        if non_null.empty:
            continue
        if (non_null % 1 == 0).all():
            df_out[col] = df_out[col].astype('Int64')

    return df_out

def build_orders_dtype(df_in):
    dtype_map = {}
    for col in df_in.columns:
        if col == 'etl_processed_timestamp':
            dtype_map[col] = TIMESTAMP()
        elif pd.api.types.is_integer_dtype(df_in[col]):
            dtype_map[col] = NUMBER(38, 0)
        elif pd.api.types.is_float_dtype(df_in[col]):
            dtype_map[col] = NUMBER(20, 6)
        else:
            max_len = int(df_in[col].astype(str).str.len().max()) if len(df_in[col]) else 1
            dtype_map[col] = VARCHAR2(max(1, min(4000, max_len + 10)))
    return dtype_map

print('[Transform] Transformation helpers are ready.')

## Step 6: Load Dimension Tables to ADW
Loads small files using `if_exists='replace'`.

In [ ]:
dimension_jobs = [
    ('cleaned_aisles.csv', 'etl_aisles'),
    ('cleaned_departments.csv', 'etl_departments'),
    ('cleaned_products.csv', 'etl_products')
]

print('[Load-Dimension] Starting dimension table loads...')
for file_name, table_name in dimension_jobs:
    print('-' * 80)
    print(f'[Load-Dimension] Reading {file_name}...')
    df = pd.read_csv(csv_paths[file_name])

    df = clean_dimension_df(df, table_name)

    print(f'[Load-Dimension] Writing to {table_name} with if_exists=replace...')
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    print(f'[Load-Dimension] Completed {table_name}: {len(df):,} rows loaded.')

print('[Load-Dimension] All dimension loads completed.')

## Step 7: Load Fact Table in Chunks
Processes `cleaned_orders.csv` with `chunksize=50000` and appends each chunk.

In [ ]:
orders_file = csv_paths['cleaned_orders.csv']
orders_table = 'etl_orders'
orders_chunksize = 50000

print('[Load-Fact] Starting chunked fact load...')
print(f'[Load-Fact] Source file: {orders_file}')
print(f'[Load-Fact] Target table: {orders_table}')
print(f'[Load-Fact] Chunk size: {orders_chunksize:,}')

chunk_iterator = pd.read_csv(orders_file, chunksize=orders_chunksize)
first_chunk = next(chunk_iterator)

first_chunk = normalize_numeric_columns(first_chunk)
if 'eval_set' in first_chunk.columns:
    first_chunk['eval_set'] = first_chunk['eval_set'].astype(str).str.strip().str.lower()
first_chunk['etl_processed_timestamp'] = pd.Timestamp.now()

orders_dtype = build_orders_dtype(first_chunk)
print('[Load-Fact] Oracle dtype mapping generated from first chunk.')

print(f'[Load-Fact] Creating/replacing empty table {orders_table}...')
first_chunk.iloc[0:0].to_sql(
    orders_table,
    con=engine,
    if_exists='replace',
    index=False,
    dtype=orders_dtype
)
print(f'[Load-Fact] Empty table {orders_table} ready.')

chunk_count = 1
total_rows_loaded = len(first_chunk)

print('-' * 80)
print('[Load-Fact] Appending chunk 1...')
first_chunk.to_sql(
    orders_table,
    con=engine,
    if_exists='append',
    index=False,
    dtype=orders_dtype
)
print(f'[Load-Fact] Chunk 1 loaded. Total rows: {total_rows_loaded:,}')

for chunk_df in chunk_iterator:
    chunk_count += 1
    chunk_rows = len(chunk_df)

    print('-' * 80)
    print(f'[Load-Fact] Processing chunk {chunk_count} with {chunk_rows:,} rows...')

    chunk_df = normalize_numeric_columns(chunk_df)
    if 'eval_set' in chunk_df.columns:
        chunk_df['eval_set'] = chunk_df['eval_set'].astype(str).str.strip().str.lower()
    chunk_df['etl_processed_timestamp'] = pd.Timestamp.now()

    chunk_df.to_sql(
        orders_table,
        con=engine,
        if_exists='append',
        index=False,
        dtype=orders_dtype
    )

    total_rows_loaded += chunk_rows
    print(f'[Load-Fact] Chunk {chunk_count} loaded. Total rows so far: {total_rows_loaded:,}')

print('-' * 80)
print('[Load-Fact] Fact table load complete.')
print(f'[Load-Fact] Total chunks processed: {chunk_count}')
print(f'[Load-Fact] Total rows loaded into {orders_table}: {total_rows_loaded:,}')

## Step 8: Validation Queries
Run checks to confirm successful load to ADW.

In [ ]:
validation_sql = """
SELECT 'etl_aisles' AS table_name, COUNT(*) AS row_count FROM etl_aisles
UNION ALL
SELECT 'etl_departments' AS table_name, COUNT(*) AS row_count FROM etl_departments
UNION ALL
SELECT 'etl_products' AS table_name, COUNT(*) AS row_count FROM etl_products
UNION ALL
SELECT 'etl_orders' AS table_name, COUNT(*) AS row_count FROM etl_orders
"""

with engine.connect() as conn:
    counts_df = pd.read_sql(text(validation_sql), conn)

print('[Validate] Row count summary:')
display(counts_df)

sample_sql = """
SELECT *
FROM etl_orders
FETCH FIRST 10 ROWS ONLY
"""

with engine.connect() as conn:
    sample_orders_df = pd.read_sql(text(sample_sql), conn)

print('[Validate] Sample rows from etl_orders:')
display(sample_orders_df)

## Step 9: Cleanup
Release database resources after completion.

In [ ]:
engine.dispose()
print('[Cleanup] SQLAlchemy engine disposed.')

## Submission Checklist (Task 6)
- Public/cloud source URLs are configured and used for extraction.
- Transform logic is clearly shown in code.
- ADW connection and load operations are successful.
- Chunked fact load logs are visible.
- Validation query outputs are captured for report screenshots.